# Solutions · Chapter 01-05 · pandas II

Worked answers with reasoning. E9 and E14 are two views of the same bug - a gap filled with a
zero, and a lag feature built on an incomplete index - and together they are the reason module 09
opens the way it does.

Self-contained: run from the top with a fresh kernel.

In [ ]:
import numpy as np
import pandas as pd

rentals = pd.DataFrame({
    "station_id": ["a", "b", "c", "a", "b", "c", "a", "b", "c", "a", "b", "c"],
    "day": ["2024-03-01"] * 3 + ["2024-03-02"] * 3 + ["2024-03-04"] * 3 + ["2024-03-05"] * 3,
    "count": [120, 80, 45, 150, 95, 60, 90, 70, 30, 200, 110, 75],
})
stations = pd.DataFrame({"station_id": ["a", "b", "c"], "site": ["north", "north", "south"],
                         "capacity": [40, 25, 15]})
stations_messy = pd.concat([stations, pd.DataFrame({"station_id": ["b"], "site": ["east"],
                                                    "capacity": [25]})], ignore_index=True)
daily = rentals.assign(day=lambda d: pd.to_datetime(d["day"]))
print("rentals", rentals.shape, "| stations", stations.shape, "| messy", stations_messy.shape)

## E1 · `groupby`, `agg` and `transform`

**Three steps:** *split* the rows into groups by a key, *apply* a function to each group, *combine*
the results.

**`agg` collapses**: one row per group. Use it when the answer is a summary - a total per station,
an error rate per segment.

**`transform` broadcasts**: one row per *original* row, carrying that row's group value. Use it
when the answer belongs back on the data - "this day's share of its station's total", "how far
above this station's average".

The test: *does my result have as many rows as the input, or as many rows as there are groups?*
That question picks the method every time.

## E2 · Why multiplication is worse than loss

**Losing rows is loud.** An inner join that drops unmatched rows changes the row count downwards,
which shows up in the very next `len()`, in every total, and usually in an obviously empty group.
Someone notices.

**Multiplying rows is quiet.** The frame is bigger, every column looks normal, and totals go up -
which, in most business data, is indistinguishable from a good month.

And the consequences are different in kind. Missing rows make you *less* informed. Duplicated rows
make you *falsely confident*: the same observation appears several times, so the model sees copies
of a row in training and is tested on its twin. The score rises, the confidence interval narrows,
and everything looks better than it is. **The bug rewards you for having it**, which is why it
survives review.

## E3 · Resample versus group-by-date

`groupby(.dt.date)` groups the rows that exist. If no row has March 3rd, no group exists, and the
result simply has no such entry.

`resample("D")` builds a **complete** index over the range - one entry per day between the first
and last - and then fills each from whatever rows fall in it. March 3rd exists as a period, and
gets whatever the aggregation of an empty set produces (`0` for `sum`, `NaN` for `mean`).

**When the difference is a bug:** whenever position implies time. Lags, rolling windows,
differences, "the previous row", plots with an evenly spaced x-axis, and any join on row order. In
all of those, "the row before" must mean "the day before", and on an incomplete index it does not.

**When it does not matter:** a total for the month, a mean by weekday, a count by station - any
answer that does not care about adjacency.

## E4 · Counting join output by hand

Left keys `[a, a, b, c, d]`, right keys `[a, b, b, e]`.

Work key by key - **each left row is repeated once per matching right row**:

| Key | In left | In right | Rows produced |
|---|---|---|---|
| a | 2 | 1 | 2 x 1 = 2 |
| b | 1 | 2 | 1 x 2 = **2** |
| c | 1 | 0 | 1 (with nulls, in a left join) |
| d | 1 | 0 | 1 (with nulls, in a left join) |
| e | 0 | 1 | 0 in left/inner; 1 in outer |

- **Left join: 6 rows.** Five in, six out - because `b` matched twice.
- **Inner join: 4 rows.** Only `a` and `b` match at all.
- **Outer join: 7 rows.** The six from the left join plus one for `e`.

**Unmatched:** `c` and `d` from the left; `e` from the right.

**The line worth memorising:** a left join returns `len(left)` rows *only if the right key is
unique*. That is exactly what `validate="many_to_one"` checks, and why a left join is not
automatically safe.

In [ ]:
left = pd.DataFrame({"k": ["a", "a", "b", "c", "d"], "v": range(5)})
right = pd.DataFrame({"k": ["a", "b", "b", "e"], "w": range(4)})
for how in ("left", "inner", "outer"):
    print(f"{how:<6} -> {len(left.merge(right, on='k', how=how))} rows")

## E5 · Differences across a gap

In [ ]:
station_a = daily[daily["station_id"] == "a"].set_index("day")["count"]
print("as stored:")
print(pd.DataFrame({"count": station_a, "diff": station_a.diff()}).to_string())

complete = station_a.asfreq("D")
print("\non a complete daily index:")
print(pd.DataFrame({"count": complete, "diff": complete.diff()}).to_string())

Mean: `(120 + 150 + 90 + 200) / 4 = 560 / 4 = **140**`.

**As stored**, the differences are `NaN, 30, -60, 110`. The `-60` is presented as a one-day drop
and is actually the change across **two** days, the 2nd to the 4th.

**On a complete index**, they are `NaN, 30, NaN, NaN, 110`. Two entries are unknown, honestly,
because you cannot compute a one-day change either into or out of a day you did not observe.

**Which would I use as a feature? The complete-index version**, without hesitation.

The stored version does not merely lose information - it *states something false*. A model trained
on it learns that a drop of 60 in one day is a thing that happens, and every downstream statistic
about daily volatility is inflated. The `NaN` version says "unknown", and unknown is a state your
pipeline can handle deliberately: drop the row, impute it with a reason, or add a "was interpolated"
flag. What you cannot handle is a wrong number that looks right.

**The general form:** *prefer a missing value to a wrong one.* It reappears in 02-04 and again as
the whole argument of 04-06.

## E6 · One row per site

In [ ]:
per_site = (rentals.merge(stations, on="station_id", how="left", validate="many_to_one")
            .groupby("site")
            .agg(station_days=("count", "size"),
                 total_rentals=("count", "sum"),
                 mean_per_day=("count", "mean"),
                 capacity=("capacity", "sum")))
per_site["rentals_per_capacity"] = (per_site["total_rentals"] / per_site["capacity"]).round(2)
print(per_site.round(2))

The last column is the one that changes the story. North takes 915 rentals against south's 210 -
more than four times as many - yet per unit of capacity they are **almost identical**, 3.52 against
3.50. North is not busier; it is bigger. It has two stations and 65 docks against south's one
station and 15.

**That is the point of the exercise, and it is not a pandas point.** A raw total mostly measures
how big something is. To compare *performance* you need a denominator, and choosing it is a
modelling decision: per dock, per station, per opening hour, per resident within 500 metres. Each
gives a different ranking and each answers a different question.

You will meet this again as normalisation in 04-06, as rate-versus-count targets in 04-01, and as
the reason a per-capita map looks nothing like a raw-count map.

(The ratio here survives the capacity bug flagged below only because every station has exactly four
days, so both numerator and denominator are inflated by the same factor of 4. That is luck, not
correctness - with unequal day counts the two sites would be scaled differently and the comparison
would be wrong.)

**One caution on `capacity=("capacity", "sum")`:** capacity is a property of a station, not of a
station-day, so summing it over four days would count each dock four times. It works here because
`.size` and `.sum` are taken over the same grouped frame and I want total docks - but this is
exactly the kind of accidental multiplication that a join creates. When an attribute is repeated
by a join, aggregate it with `first` or deduplicate before summing, and check that the number is
what a human would say.

In [ ]:
# The careful version: capacity summed over distinct stations, not over station-days.
docks = stations.groupby("site")["capacity"].sum()
print("docks per site (correct):")
print(docks.to_string())
print("\nnaive sum over station-days would give:", 
      rentals.merge(stations, on='station_id').groupby('site')['capacity'].sum().tolist(),
      "- four times too big")

## E7 · A merge that checks itself

In [ ]:
def safe_merge(left, right, on, how="left", expect="many_to_one"):
    """Merge, then verify the row count behaved as the join type promises."""
    before = len(left)
    out = left.merge(right, on=on, how=how, validate=expect, indicator=True)
    unmatched = int((out["_merge"] == "left_only").sum())
    if how == "left":
        assert len(out) == before, f"left join changed the row count: {before} -> {len(out)}"
    if unmatched:
        print(f"  note: {unmatched} of {before} rows found no match on {on!r}")
    return out.drop(columns="_merge")


good = safe_merge(rentals, stations, on="station_id")
print("clean lookup ok:", good.shape)

try:
    safe_merge(rentals, stations_messy, on="station_id")
except Exception as exc:
    print("messy lookup rejected:", type(exc).__name__, "-", str(exc).splitlines()[0])

Both defences in one function: `validate` catches a non-unique key at the moment of the merge, and
the assert catches a row-count change even for join types `validate` does not cover.

**Why report unmatched rows rather than assert on them.** Unmatched rows are often legitimate - a
new station not yet in the reference table, a customer who has not been enriched. Failing on them
would make the function unusable; saying nothing would let a join that matched 3% of rows pass
silently. Printing the count puts the judgement where it belongs, with the person reading the
output.

**The pattern generalises.** A wrapper that does the operation *and* checks the invariant is worth
writing for anything you do repeatedly and that can fail quietly: merges, loads, resamples,
train/test splits. It costs ten lines once and removes a class of mistake permanently.

## E8 · What duplication distorts

| Statistic | Effect | Why |
|---|---|---|
| **Sum / count** | Badly distorted (1125 -> 1480) | Duplicated rows are counted again |
| **Mean** | Barely moved (93.75 -> 92.50) | It is a *weighted* mean now, over-weighting the duplicated group. It moves only as far as that group differs from the rest |
| **Max / min** | Unchanged (200, 30) | Copying a value cannot create a new extreme |
| **Median / quantiles** | Shifted toward the duplicated group | The group now occupies more of the distribution |
| **Standard deviation** | Understated | Repeated identical values look like consistency |
| **A model's test score** | **Inflated, sometimes enormously** | Copies of a row land on both sides of a split |

**Why "the mean barely moved" is not reassurance - it is the problem.**

The mean moved by 1.25 because station b happens to be close to average. Had the duplicated station
been the busiest or the quietest, the same corruption would have moved it much further. **The size
of the error depends on which row was duplicated, not on how serious the bug is.** A statistic that
sometimes hides a fault and sometimes does not is no evidence at all.

And the statistic that matters most - the model's held-out score - is the one this bug inflates
hardest, precisely because duplicates make the test set a partial copy of the training set. A
sanity check on the mean would have found nothing while the model's reported accuracy was inflated
by leakage. **Check the row count, not the summary statistics.**

## E9 · "We fixed the gaps by filling with zero"

**What probably happened.** The gaps were not days with no rentals - they were days with no
*data*: the stand was closed, a sensor was down, an export failed. Filling them with `0` inserted
fabricated observations of very low demand. Every model that uses recent history then learns from
days that never happened: a lag feature reports "yesterday was a total washout" when yesterday was
ordinary, rolling averages are dragged down around every gap, and if the gaps cluster - a
maintenance week, a holiday shutdown - the model learns a seasonal pattern that is an artefact of
the export.

**Two ways to confirm it:**

1. **Check whether the zeros coincide with the gaps.** Compare the dates that are zero after the
   fix with the dates that were absent before it. If they are the same set, the zeros are
   manufactured, not measured. If real zero-demand days exist elsewhere in the history, compare:
   genuine zeros will be scattered, manufactured ones will sit exactly on the old holes.
2. **Look at the error by date.** Slice the model's errors around each filled day. If the damage
   is concentrated in the one to seven days *following* each fill - the reach of the lag and
   rolling features - that is the mechanism, visibly.

**The right fix.** Reindex to a complete daily index so the gaps are *explicit*, mark them `NaN`,
and then decide what they mean - which requires asking why the days are missing. Depending on the
answer: exclude those rows from training, carry an "imputed" indicator column so the model can
learn that those days are different, or interpolate with a stated method. And regardless of the
choice, do not let a lag feature reach across a gap without knowing it did.

**The lesson.** "Fill the gaps" sounds like cleaning. It is *inventing data*, and it is only
cleaning if the invented value is what actually happened.

## E10 · "Surely you know your own data"

> I know what the data is supposed to be; the check is about what arrived. A duplicate in a
> reference table can appear at any time - a station relocated, a customer merged, a slowly
> changing dimension that gained a second version - and it changes my results without changing my
> code or raising anything. The check is one line and it converts an entire class of silent
> corruption into an immediate, loud failure at the guilty line. And of all the ways to be wrong,
> this is the one that makes a model look *better*, so it is the one least likely to be caught
> downstream.

**What the interviewer is listening for:** that you distinguish *the schema you were promised* from
*the file in front of you*, and that you know duplication inflates model scores rather than
degrading them.

## E11 · 10 million transactions, uncertain customer ids

**In this order, because each step is cheaper than the next:**

1. **Check uniqueness on the right side alone**, before any join:
   `customers["customer_id"].duplicated().sum()`. One pass, no join, and it answers the whole
   question.
2. **If there are duplicates, look at them.** `customers[customers.duplicated("customer_id",
   keep=False)].sort_values("customer_id").head(40)`. You need to know *why* they exist, and the
   rows will usually tell you: two addresses, a historical record with a validity date, a merged
   account, or a genuine data-entry duplicate.
3. **Count the blast radius.** How many transactions reference a duplicated id? If it is 12 rows
   out of 10 million, the fix is different from 3 million.
4. **Only then join**, with `validate=` set to whatever the truth turned out to be.

**If the ids are not unique**, the fix depends on the reason, and this is a data-modelling decision
rather than a pandas one:

- **Duplicates are genuinely redundant** - drop them, `drop_duplicates("customer_id")`, after
  checking the other columns actually agree.
- **They are versions over time** - the join key is not `customer_id`, it is `customer_id` plus a
  date range, and you need an as-of join (`pd.merge_asof`) so each transaction gets the record
  that was current *at the time*. Using today's record for a two-year-old transaction is
  information from the future, which is leakage (04-05).
- **They are genuinely different entities sharing an id** - the upstream system is broken and no
  join can be correct. Escalate rather than choose.

**What I would not do:** silently `drop_duplicates` to make the join run. That picks an arbitrary
row - whichever happened to be first - and buries a decision nobody made.

## E12 · Admissions and lab results

**Aggregate first, then join.**

1. `worst = labs.groupby("admission_id")["value"].max()` - collapses many lab rows to **one row per
   admission**.
2. `admissions.merge(worst, on="admission_id", how="left", validate="one_to_one")` - the row count
   stays exactly `len(admissions)`.

**Row counts:** labs has many rows per admission; after the `groupby` it has at most one per
admission; after the left join, admissions is unchanged in length, with `NaN` for admissions that
had no labs.

**What goes wrong in the other order.** Joining first gives one row *per lab result*, so a patient
with forty tests appears forty times. If you then take the max per admission you get back to where
you should have been - but if you forget, or if a later step aggregates before you collapse, you
have silently weighted every patient by how many tests they had. Sicker patients get more tests, so
that weighting is not random: it systematically over-represents exactly the patients whose outcomes
differ. Every downstream average is now about "per test" instead of "per patient", and the two
questions have different answers.

**And the timing question that decides whether any of this is legitimate:** *when were those labs
taken?* If the model predicts something at admission, only labs from before that moment may be
used. `max()` over the whole stay quietly includes results from after the outcome was determined,
which is textbook target leakage - 04-05.

## E13 · Explaining it to Maria

> A zero says "we were open and nobody came". A blank says "we do not know what happened". Those
> are different facts, and the forecast learns from both. By writing zeros into the days when the
> counter was broken, we taught it that trade collapses now and then for no reason - so now it
> expects collapses that never happened.

(58 words.)

**Where the explanation stops being complete:** it might sound as though a blank is always the
right answer. It is not - if the stand really was open and rented nothing, `0` is the truth and
recording a blank would throw away a real observation. The point is that the two cases look
identical in the file, so somebody has to know which happened. That is a question for whoever ran
the stand, not for the analyst.

## E14 · A lag feature, built twice

In [ ]:
# Version 1: lag on the data as stored, one shift per station.
as_stored = daily.sort_values(["station_id", "day"]).copy()
as_stored["lag_1"] = as_stored.groupby("station_id")["count"].shift(1)

# Version 2: lag on a COMPLETE daily index per station.
full_index = pd.MultiIndex.from_product(
    [sorted(daily["station_id"].unique()), pd.date_range(daily["day"].min(), daily["day"].max(), freq="D")],
    names=["station_id", "day"])
complete = (daily.set_index(["station_id", "day"]).reindex(full_index).reset_index())
complete["lag_1"] = complete.groupby("station_id")["count"].shift(1)

comparison = (as_stored.merge(complete, on=["station_id", "day"], suffixes=("_stored", "_correct"))
              [["station_id", "day", "count_stored", "lag_1_stored", "lag_1_correct"]])
print(comparison.to_string(index=False))

In [ ]:
wrong_rows = comparison[comparison["lag_1_stored"].notna()
                        & (comparison["lag_1_stored"] != comparison["lag_1_correct"].fillna(-1))]
print("rows where the stored lag is wrong:", len(wrong_rows), "of", len(comparison))
print(wrong_rows.to_string(index=False))

**Three rows are wrong - one per station, and always the same day: March 4th**, the day after the
gap.

On the stored data, "the previous row" for March 4th is **March 2nd**, so the feature reports the
2nd's count and calls it yesterday. On the complete index, the previous row is March 3rd, which is
missing, so the lag is correctly `NaN`: we do not know what yesterday was.

**How a model would be misled.** For station a the feature says yesterday was 150 when the honest
answer is unknown - and 150 was a *busy* day, while the actual value on the 4th was 90. The model
learns "a busy yesterday is followed by a quiet today" from a pair of days that were never adjacent.
Three of twelve rows carry a fabricated relationship. In a real series with a dozen gaps a year, the
model absorbs a dozen invented transitions, and no diagnostic will point at them - the feature is a
plausible number, the model trains normally, and the error shows up only as unexplained inaccuracy.

**How to build it so the bug cannot happen:**

1. **Reindex to a complete index first, always**, per group - `date_range` crossed with the group
   keys, exactly as above. Do this before any shift, diff, rolling window or `pct_change`.
2. **Assert the index is complete** before creating lags:
   `assert complete.groupby("station_id")["day"].size().eq(expected_days).all()`.
3. **Never let a shift cross a group boundary.** `groupby(...).shift(1)` is right; a bare
   `shift(1)` on a frame sorted by station would hand station b's first day the last value from
   station a - a different version of the same bug, and even harder to see.
4. **Decide what the `NaN`s mean** and record the decision, rather than letting a later
   `dropna()` quietly delete the rows after every gap.

Module 09 builds all of this properly, with backtesting to prove it. You now know why it insists.

---

## Where to go next

Back to the chapter for the mastery check and flashcards, then **01-06 · Plotting that says
something, and reproducible randomness**, which closes module 01.